In [1]:
import dash
from dash import dcc, html,Dash
from dash.dependencies import Input, Output
import plotly.express as px
import pandas as pd
import numpy as np


In [2]:
# Load your preprocessed data
evplopulation_preprocessed = pd.read_csv('cleaned_evpopulation_data.csv')

In [11]:
evplopulation_preprocessed.head()

,index,Date,County,State,Vehicle Primary Use,Battery Electric Vehicles (BEVs),Plug-In Hybrid Electric Vehicles (PHEVs),Electric Vehicle (EV) Total,Non-Electric Vehicle Total,Total Vehicles,...,BEV_in_West,Winter_EV,EV_Growth_Rate,Market_Growth_Rate,EV_Z_Score,BEV_Percentage,PHEV_Percentage,EV_to_NonEV_Ratio,Dominant_EV_Type,West_BEV
0,0,2018-01-31,Brevard,FL,Passenger,0,1,1,109,110,...,NaN,1,NaN,NaN,-0.337146,0.00,0.91,0.009174,PHEV,0
1,1,2021-03-31,Pinellas,FL,Passenger,1,1,2,113,115,...,NaN,0,NaN,NaN,-0.258512,0.87,0.87,0.017699,Equal,0
2,2,2020-08-31,Shasta,CA,Passenger,1,0,1,36,37,...,1.0,0,NaN,NaN,-0.167228,2.70,0.00,0.027778,BEV,1
3,3,2020-06-30,Bucks,PA,Passenger,1,0,1,24,25,...,NaN,0,NaN,NaN,-0.044329,4.00,0.00,0.041667,BEV,0
4,4,2023-02-28,Snohomish,WA,Passenger,10907,2828,13735,528837,542572,...,10907.0,13735,NaN,NaN,-0.183451,2.01,0.52,0.025972,BEV,1


In [7]:
# Initialize Dash App
app = Dash(__name__)

## App Layout
app.layout = html.Div([
    html.H1("Electric Vehicle  Dashboard", style={'textAlign': 'center'}),
    
    # Filters Row
    html.Div([
        html.Div([
            html.Label("Select Year Range:"),
            dcc.RangeSlider(
                id='year-slider',
                min=evplopulation_preprocessed['Year'].min(),
                max=evplopulation_preprocessed['Year'].max(),
                value=[evplopulation_preprocessed['Year'].min(), evplopulation_preprocessed['Year'].max()],
                marks={str(year): str(year) for year in evplopulation_preprocessed['Year'].unique()},
                step=None
            )
        ], style={'width': '48%', 'display': 'inline-block'}),
        
        html.Div([
            html.Label("Select Region:"),
            dcc.Dropdown(
                id='region-dropdown',
                options=[{'label': r, 'value': r} for r in evplopulation_preprocessed['Region'].unique()],
                value=evplopulation_preprocessed['Region'].unique(),
                multi=True
            )
        ], style={'width': '48%', 'float': 'right', 'display': 'inline-block'})
    ]),
    
    # Main Charts Row
    html.Div([
        html.Div([
            dcc.Graph(id='adoption-trend')
        ], style={'width': '48%', 'display': 'inline-block'}),
        
        html.Div([
            dcc.Graph(id='regional-comparison')
        ], style={'width': '48%', 'float': 'right', 'display': 'inline-block'})
    ]),
    
    # Secondary Charts Row
    html.Div([
        html.Div([
            dcc.Graph(id='technology-mix')
        ], style={'width': '48%', 'display': 'inline-block'}),
        
        html.Div([
            dcc.Graph(id='fleet-size-impact')
        ], style={'width': '48%', 'float': 'right', 'display': 'inline-block'})
    ]),
    
    # Data Table
    html.Div([
        html.H3("Detailed Data"),
        html.Div(id='data-table')
    ])
])

@app.callback(
    [Output('adoption-trend', 'figure'),
     Output('regional-comparison', 'figure'),
     Output('technology-mix', 'figure'),
     Output('fleet-size-impact', 'figure'),
     Output('data-table', 'children')],
    [Input('year-slider', 'value'),
     Input('region-dropdown', 'value')]
)
def update_dashboard(selected_years, selected_regions):
    # Filter data
    filtered_df = evplopulation_preprocessed[
        (evplopulation_preprocessed['Year'] >= selected_years[0]) & 
        (evplopulation_preprocessed['Year'] <= selected_years[1]) & 
        (evplopulation_preprocessed['Region'].isin(selected_regions))
    ]
    
    # 1. Adoption Trend Chart
    trend_fig = px.line(
        filtered_df.groupby(['Year', 'Month'])['Percent Electric Vehicles'].mean().reset_index(),
        x='Month',
        y='Percent Electric Vehicles',
        color='Year',
        title="Monthly EV Adoption Trend",
        labels={'Percent Electric Vehicles': 'EV Adoption %'}
    )
    
    # 2. Regional Comparison Chart
    regional_fig = px.bar(
        filtered_df.groupby('Region')['Percent Electric Vehicles'].mean().reset_index(),
        x='Region',
        y='Percent Electric Vehicles',
        color='Region',
        title="Average EV Adoption by Region"
    )
    
    # 3. Technology Mix Chart
    tech_fig = px.pie(
        filtered_df.groupby('Dominant_EV_Type').size().reset_index(name='count'),
        values='count',
        names='Dominant_EV_Type',
        title="Dominant EV Technology Mix"
    )
    
    # 4. Fleet Size Impact Chart - CORRECTED
    fleet_fig = px.box(
        filtered_df,
        x='Vehicle_Size_Bin',
        y='Percent Electric Vehicles',
        color='Vehicle_Size_Bin',
        title="EV Adoption by Vehicle Size Category"
    )
    
    # 5. Data Table
    table = dash.dash_table.DataTable(
        columns=[{"name": i, "id": i} for i in filtered_df.columns],
        data=filtered_df.to_dict('records'),
        page_size=10,
        style_table={'overflowX': 'auto'}
    )
    
    return trend_fig, regional_fig, tech_fig, fleet_fig, table
if __name__ == '__main__':
    #app.run(jupyter_mode='inline', debug=True)  
    app.run(jupyter_mode='external', port=8051)# debug=True helps see errors

Dash app running on http://127.0.0.1:8051/
